# SAM3-I：GitHub 自动部署、模型常驻显存、多次单图推理与编号展示

只需本 notebook、自己的 Google Drive 权重和一张图片。代码自动从 [GitHub 仓库](https://github.com/xheihex/SAM3-I) 下载；不需要上传源码 ZIP。作者的模型源码保持原样，沿用已验证的单图推理脚本。

2026-11-03 提供的运行记录已确认：Colab metadata 为 A100，BF16 / simple / 阈值 0.6 输出 1 个实例，PyTorch 峰值已分配显存约 7.64 GiB。该记录尚未执行最后的云盘保存步骤，也未完整留存子进程日志和实例分数。新版将保存日志，并在推理完成的同一单元格自动保存结果到 Drive。

先选择 GPU 运行时，再按顺序执行。图上显示框、半透明掩码和从 **1** 开始的编号；**图下方显示每个编号的置信度**。默认提示沿用已验证的简单指令。

模型只加载一次；之后可重复换提示词或上传新图推理。每次仍自动保存结果到 Drive。运行时被回收或重置后需重新部署。

## 1. 从 GitHub 下载源码

先运行两个折叠的辅助单元格，再运行下载单元格。它们仅定义部署与展示函数，不加载模型。默认下载 `main`；需要重现某一版本时可将 `GIT_REF` 改为完整提交号。每次创建独立 checkout，实际提交号会随结果保存。

In [ ]:
# @title 部署辅助函数（运行一次）
"""Colab deployment helpers, mirrored into the standalone notebook."""
from __future__ import annotations

import hashlib
import json
import os
from pathlib import Path
import shutil
import subprocess
from datetime import datetime, timezone


def run_logged(command, log_path, *, env=None, cwd=None, echo=True):
    """Print subprocess output into the notebook and keep the same text on disk."""
    log_path = Path(log_path)
    log_path.parent.mkdir(parents=True, exist_ok=True)
    with log_path.open("w", encoding="utf-8") as log:
        log.write(json.dumps([str(arg) for arg in command], ensure_ascii=False) + "\n")
        log.flush()
        with subprocess.Popen(
            command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
            text=True, encoding="utf-8", errors="replace", bufsize=1, env=env, cwd=cwd,
        ) as process:
            for line in process.stdout:
                if echo:
                    print(line, end="", flush=True)
                log.write(line)
                log.flush()
            returncode = process.wait()
    if returncode:
        print(f"Command failed; log: {log_path}", flush=True)
        raise subprocess.CalledProcessError(returncode, command)
    return log_path


def checkout_github(repo_url, ref, destination, logs_dir):
    """Download a branch/tag/commit into a fresh, detached Git checkout."""
    destination = Path(destination)
    if destination.exists():
        raise FileExistsError(f"Use a new checkout directory: {destination}")
    if not ref.strip() or ref.startswith("-"):
        raise ValueError("Git ref must be a nonempty branch, tag, or commit.")
    logs_dir = Path(logs_dir)
    run_logged(["git", "init", str(destination)], logs_dir / "git-init.log")
    run_logged(["git", "-C", str(destination), "remote", "add", "origin", repo_url],
               logs_dir / "git-remote.log")
    run_logged(["git", "-C", str(destination), "fetch", "--depth", "1", "origin", ref],
               logs_dir / "git-fetch.log")
    run_logged(["git", "-C", str(destination), "checkout", "--detach", "FETCH_HEAD"],
               logs_dir / "git-checkout.log")
    commit = subprocess.check_output(
        ["git", "-C", str(destination), "rev-parse", "HEAD"], text=True,
    ).strip()
    required = ("pyproject.toml", "uv.lock", ".python-version",
                "scripts/colab_single_image.py", "scripts/audit_single_image_checkpoint.py",
                "sam3/assets/bpe_simple_vocab_16e6.txt.gz")
    for name in required:
        if not (destination / name).is_file():
            raise FileNotFoundError(f"GitHub revision lacks required file: {name}")
    source = {
        "repository": repo_url, "requested_ref": ref, "git_commit": commit,
        "downloaded_at_utc": datetime.now(timezone.utc).isoformat(),
        "runtime_file_sha256": {
            name: hashlib.sha256((destination / name).read_bytes()).hexdigest()
            for name in required
        },
    }
    print(f"GitHub revision: {commit}", flush=True)
    return destination, source


def ensure_local_checkpoint(source, destination):
    """Reuse a complete local copy only while the source metadata still matches."""
    source, destination = Path(source), Path(destination)
    if not source.is_file():
        raise FileNotFoundError(f"Checkpoint not found: {source}")
    source_stat = source.stat()
    fingerprint = {
        "source_path": str(source.resolve()), "size_bytes": source_stat.st_size,
        "mtime_ns": source_stat.st_mtime_ns,
    }
    if source.resolve() == destination.resolve():
        return fingerprint
    destination.parent.mkdir(parents=True, exist_ok=True)
    metadata_path = destination.with_suffix(destination.suffix + ".source.json")
    if destination.is_file() and metadata_path.is_file():
        previous = json.loads(metadata_path.read_text())
        if previous == fingerprint and destination.stat().st_size == source_stat.st_size:
            print("Reusing local checkpoint; Drive source metadata is unchanged.", flush=True)
            return fingerprint
    if shutil.disk_usage(destination.parent).free < source_stat.st_size + 1024**3:
        raise RuntimeError("Insufficient local disk space for checkpoint copy + 1 GiB.")
    partial = destination.with_suffix(destination.suffix + ".part")
    print("Copying checkpoint from Drive to local disk...", flush=True)
    shutil.copy2(source, partial)
    if partial.stat().st_size != source_stat.st_size:
        raise RuntimeError("Incomplete local checkpoint copy.")
    partial.replace(destination)
    metadata_path.write_text(json.dumps(fingerprint, indent=2), encoding="utf-8")
    return fingerprint


def render_saved_results(output_dir, image_path, source, overlay_renderer, figure_renderer):
    """Add stable IDs and presentation artifacts to old or new inference outputs."""
    import numpy as np
    from PIL import Image

    output_dir = Path(output_dir)
    result_path = output_dir / "result.json"
    result = json.loads(result_path.read_text())
    predictions = result["predictions"]
    with Image.open(image_path) as opened:
        image = opened.convert("RGB")
    if (result["width"], result["height"]) != image.size:
        raise ValueError("Result dimensions do not match the uploaded image.")
    masks = np.empty((len(predictions), image.height, image.width), dtype=bool)
    for index, prediction in enumerate(predictions):
        # Keep the saved prediction order and mask filenames; only add one-based IDs.
        prediction["instance_id"] = index + 1
        mask_path = (output_dir / prediction["mask_file"]).resolve()
        if not mask_path.is_relative_to(output_dir.resolve()):
            raise ValueError("Mask file must be inside the result directory.")
        with Image.open(mask_path) as opened_mask:
            if opened_mask.size != image.size:
                raise ValueError("Mask dimensions do not match the uploaded image.")
            masks[index] = np.asarray(opened_mask.convert("L")) > 0
    overlay = overlay_renderer(image, masks, predictions)
    overlay.save(output_dir / "overlay.png")
    figure = figure_renderer(overlay, predictions)
    figure.save(output_dir / "result_figure.png")
    result["source"] = source
    result["visualization"] = {
        "instance_id_start": 1, "overlay": "overlay.png", "figure": "result_figure.png",
        "description": "Colored masks, boxes and IDs; confidence scores below the image.",
    }
    result_path.write_text(json.dumps(result, ensure_ascii=False, indent=2), encoding="utf-8")
    return result, figure


def archive_run(output_dir, source, audit_path, logs_dir, environment_path, drive_results):
    """Save provenance and logs with results, then immediately persist a ZIP to Drive."""
    output_dir, drive_results = Path(output_dir), Path(drive_results)
    (output_dir / "source-manifest.json").write_text(
        json.dumps(source, ensure_ascii=False, indent=2), encoding="utf-8",
    )
    shutil.copy2(audit_path, output_dir / "checkpoint-audit.json")
    shutil.copy2(environment_path, output_dir / "environment.json")
    shutil.copytree(logs_dir, output_dir / "logs", dirs_exist_ok=True)
    archive = Path(shutil.make_archive(str(output_dir), "zip", root_dir=output_dir))
    drive_results.mkdir(parents=True, exist_ok=True)
    saved_archive = drive_results / archive.name
    shutil.copy2(archive, saved_archive)
    print(f"Saved to Drive: {saved_archive}", flush=True)
    return archive, saved_archive


class ResidentInferenceWorker:
    """Control a direct Python child with file replies and unparsed stdout logs."""

    def __init__(self, command, ipc_dir, log_path, *, env=None, cwd=None, startup_timeout=600):
        import atexit
        self.ipc_dir = Path(ipc_dir)
        self.ipc_dir.mkdir(parents=True, exist_ok=False)
        self.log_path = Path(log_path)
        self.log_path.parent.mkdir(parents=True, exist_ok=True)
        self._log_offset = 0
        self._busy = False
        self._log_writer = self.log_path.open("w", encoding="utf-8")
        child_env = dict(os.environ if env is None else env, PYTHONUNBUFFERED="1")
        try:
            self.process = subprocess.Popen(
                command, stdin=subprocess.PIPE, stdout=self._log_writer,
                stderr=subprocess.STDOUT, text=True, encoding="utf-8",
                env=child_env, cwd=cwd,
            )
        except BaseException:
            self._log_writer.close()
            raise
        try:
            ready = self._wait("ready", startup_timeout)
            if not ready["ok"]:
                raise RuntimeError(ready.get("error", "Worker startup failed."))
            self.info = ready["result"]
        except BaseException:
            self.stop()
            raise
        atexit.register(self.stop)

    def is_alive(self):
        return self.process.poll() is None

    def _show_new_logs(self):
        with self.log_path.open(encoding="utf-8", errors="replace") as stream:
            stream.seek(self._log_offset)
            chunk = stream.read()
            self._log_offset = stream.tell()
        if chunk:
            print(chunk, end="", flush=True)

    def _wait(self, name, timeout):
        import time
        deadline = time.monotonic() + timeout
        reply_path = self.ipc_dir / (name + ".json")
        while True:
            self._show_new_logs()
            # Prefer the final reply even when a fatal error just stopped the host.
            if reply_path.is_file():
                reply = json.loads(reply_path.read_text(encoding="utf-8"))
                reply_path.unlink()
                return reply
            if not self.is_alive():
                self._show_new_logs()
                raise RuntimeError(
                    f"Model worker exited ({self.process.returncode}); "
                    f"see {self.log_path}. Run the model loading cell again."
                )
            if time.monotonic() >= deadline:
                raise TimeoutError(f"Model worker timed out; see {self.log_path}.")
            time.sleep(0.1)

    def predict(self, request, *, log_path=None, timeout=600):
        import uuid
        if not self.is_alive():
            raise RuntimeError("Model worker is stopped. Run the model loading cell again.")
        if self._busy:
            raise RuntimeError("A query is already running in this model worker.")
        self._busy = True
        offset = self.log_path.stat().st_size
        try:
            request_id = uuid.uuid4().hex
            self.process.stdin.write(json.dumps({**request, "id": request_id}, ensure_ascii=False) + "\n")
            self.process.stdin.flush()
            reply = self._wait(request_id, timeout)
            if not reply["ok"]:
                if reply.get("fatal"):
                    self.stop()
                raise RuntimeError(
                    f"{reply.get('error_type', 'Error')}: {reply['error']}. "
                    f"Worker log: {self.log_path}"
                )
            return reply["result"]
        except (KeyboardInterrupt, TimeoutError, BrokenPipeError):
            # Do not leave an unknown request running, or silently queue a retry.
            self.stop()
            raise
        finally:
            self._busy = False
            if log_path is not None:
                destination = Path(log_path)
                destination.parent.mkdir(parents=True, exist_ok=True)
                with self.log_path.open("rb") as stream:
                    stream.seek(offset)
                    destination.write_bytes(stream.read())

    def stop(self):
        """Stop/reap the actual GPU Python process, releasing its CUDA context."""
        if self.is_alive():
            self.process.terminate()
            try:
                self.process.wait(timeout=5)
            except subprocess.TimeoutExpired:
                self.process.kill()
                self.process.wait(timeout=5)
        else:
            self.process.wait()
        if self.process.stdin and not self.process.stdin.closed:
            self.process.stdin.close()
        if not self._log_writer.closed:
            self._log_writer.close()


def start_model_worker(repo, checkpoint, runtime_dir, logs_dir, *,
                       env=None, worker_source=None, existing=None):
    """Reuse an identical live model, or stop it before loading a replacement."""
    import uuid
    repo, checkpoint = Path(repo).resolve(), Path(checkpoint).resolve()
    bpe = repo / "sam3/assets/bpe_simple_vocab_16e6.txt.gz"
    if worker_source is None:
        worker_source = Path(__file__).with_name("colab_inference_worker.py").read_text(encoding="utf-8")
    def identity(path):
        stat = path.stat()
        return [str(path), stat.st_size, stat.st_mtime_ns]
    config = {
        "repo": str(repo), "checkpoint": identity(checkpoint), "bpe": identity(bpe),
        "worker_sha256": hashlib.sha256(worker_source.encode("utf-8")).hexdigest(),
        "adapter_sha256": hashlib.sha256((repo / "scripts/colab_single_image.py").read_bytes()).hexdigest(),
    }
    if existing is not None:
        if existing.is_alive() and getattr(existing, "model_config", None) == config:
            print(f"Reusing resident model: {existing.info['worker_id']}", flush=True)
            return existing
        existing.stop()
    runtime_dir, logs_dir = Path(runtime_dir), Path(logs_dir)
    runtime_dir.mkdir(parents=True, exist_ok=True)
    host_id = uuid.uuid4().hex[:12]
    worker_script = runtime_dir / f"worker-{host_id}.py"
    worker_script.write_text(worker_source, encoding="utf-8")
    ipc_dir = runtime_dir / f"ipc-{host_id}"
    # uv already synchronized this environment. Launch Python directly so stop()
    # always owns the CUDA process rather than a uv wrapper.
    python = repo / ".venv/bin/python"
    if not python.is_file():
        raise FileNotFoundError("Run the uv environment preparation cell first.")
    worker = ResidentInferenceWorker(
        [str(python), str(worker_script), "--repo-root", str(repo),
         "--checkpoint", str(checkpoint), "--bpe", str(bpe), "--ipc-dir", str(ipc_dir)],
        ipc_dir, logs_dir / f"model-worker-{host_id}.log", env=env, cwd=repo,
    )
    worker.model_config = config
    return worker

EMBEDDED_SOURCE_SHA256 = dict(globals().get("EMBEDDED_SOURCE_SHA256", {}))
EMBEDDED_SOURCE_SHA256['scripts/colab_notebook_helpers.py'] = 'fb8a6902abaeb46a7dec905c2f946216f9acb3c0f570368c0aefa4106e4159ec'
COLAB_WORKER_SOURCE = '#!/usr/bin/env python3\n"""Persistent SAM3-I model host for the Colab adapter; no upstream changes."""\nfrom __future__ import annotations\n\nimport argparse\nimport gc\nimport json\nimport os\nfrom pathlib import Path\nimport sys\nimport time\nimport traceback\nimport uuid\n\nimport torch\n\n# Filled after --repo-root is parsed when launched outside the checkout.\nsingle = None\n\n\nclass ResidentSession:\n    """Keep only the eval model between requests, never an image/prompt state."""\n\n    def __init__(self, checkpoint, bpe):\n        self.checkpoint, self.bpe = Path(checkpoint), Path(bpe)\n        self.worker_id = uuid.uuid4().hex\n        self.query_index = 0\n        if not torch.cuda.is_available():\n            raise RuntimeError("Enable a CUDA GPU runtime before loading the model.")\n        self.native_bf16 = torch.cuda.is_bf16_supported(including_emulation=False)\n        started = time.perf_counter()\n        self.model = single.build_model(self.checkpoint, self.bpe)\n        torch.cuda.synchronize()\n        self.model_load_seconds = time.perf_counter() - started\n        self.info = {\n            "worker_id": self.worker_id, "worker_pid": os.getpid(),\n            "model_load_seconds": self.model_load_seconds,\n            "checkpoint": str(self.checkpoint), "gpu": torch.cuda.get_device_name(0),\n            "native_bf16": self.native_bf16, "torch": torch.__version__,\n        }\n\n    def run(self, request):\n        image_path = Path(request["image"])\n        prompt, category = request["prompt"], request["category"]\n        threshold = float(request["threshold"])\n        single.validate_request(image_path, prompt, category, threshold)\n        requested = request.get("precision", "auto")\n        if requested not in ("auto", "bf16", "fp16", "fp32"):\n            raise ValueError("Unknown precision.")\n        precision = single.choose_precision(requested, self.native_bf16)\n        output = Path(request["output_dir"])\n        if output.exists() and (not output.is_dir() or any(output.iterdir())):\n            raise ValueError("Output directory is not empty; use a new directory per query.")\n        image = single.open_rgb(image_path)\n        self.query_index += 1\n        torch.cuda.synchronize()\n        torch.cuda.reset_peak_memory_stats()\n        started = time.perf_counter()\n        # Fresh processor/image/text state per query; only self.model stays resident.\n        state = single.predict(\n            self.model, image, prompt, category, threshold,\n            device="cuda", precision=precision,\n        )\n        torch.cuda.synchronize()\n        inference_seconds = time.perf_counter() - started\n        peak_gib = torch.cuda.max_memory_allocated() / 2**30\n        result = single.save_results(image, state, output, {\n            "image_file": str(image_path), "prompt": prompt, "category": category,\n            "text_type": single.TEXT_TYPES[category], "threshold": threshold,\n            "precision": precision, "checkpoint": str(self.checkpoint), "inst_stage": "3",\n            "worker_id": self.worker_id, "worker_pid": os.getpid(),\n            "query_index": self.query_index, "model_load_seconds": self.model_load_seconds,\n            "inference_seconds": inference_seconds, "elapsed_seconds": inference_seconds,\n            "timing_scope": "Per-query predict including CPU mask resize; excludes model load, export and display.",\n            "peak_cuda_allocated_gib": peak_gib,\n        })\n        # Do not return state/backbone_out: these contain GPU tensors.\n        return result\n\n\ndef write_reply(ipc_dir, name, reply):\n    destination = ipc_dir / (name + ".json")\n    partial = destination.with_suffix(".tmp")\n    partial.write_text(json.dumps(reply, ensure_ascii=False), encoding="utf-8")\n    partial.replace(destination)\n\n\ndef serve(session, ipc_dir):\n    write_reply(ipc_dir, "ready", {"ok": True, "result": session.info})\n    print(json.dumps({"model_ready": session.info}, ensure_ascii=False), flush=True)\n    for line in sys.stdin:\n        request = json.loads(line)\n        request_id = request.pop("id")\n        if not isinstance(request_id, str) or len(request_id) != 32:\n            raise ValueError("Invalid request ID.")\n        int(request_id, 16)\n        print(json.dumps({"request_id": request_id, **request}, ensure_ascii=False), flush=True)\n        try:\n            result = session.run(request)\n        except Exception as error:\n            # CUDA runtime/OOM errors invalidate this host. Validation/I/O errors\n            # can be corrected without paying for another model load.\n            fatal = isinstance(error, (torch.OutOfMemoryError, RuntimeError))\n            reply = {\n                "ok": False, "error": str(error), "error_type": type(error).__name__,\n                "fatal": fatal,\n            }\n            traceback.print_exc()\n        else:\n            reply = {"ok": True, "result": result}\n        # Exception traceback and run() locals are out of scope here.\n        if not reply["ok"] and not reply.get("fatal"):\n            gc.collect()\n            try:\n                torch.cuda.empty_cache()\n            except Exception:\n                # A broken CUDA context must not hide the original request error.\n                traceback.print_exc()\n                reply["fatal"] = True\n        write_reply(ipc_dir, request_id, reply)\n        print(f"Request {request_id}: {\'ok\' if reply[\'ok\'] else \'failed\'}", flush=True)\n        if reply.get("fatal"):\n            return\n\n\ndef main():\n    global single\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--repo-root", type=Path, required=True)\n    parser.add_argument("--checkpoint", type=Path, required=True)\n    parser.add_argument("--bpe", type=Path)\n    parser.add_argument("--ipc-dir", type=Path, required=True)\n    args = parser.parse_args()\n    sys.path.insert(0, str(args.repo_root / "scripts"))\n    import colab_single_image as adapter\n    single = adapter\n    args.ipc_dir.mkdir(parents=True, exist_ok=True)\n    session = ResidentSession(args.checkpoint, args.bpe or single.DEFAULT_BPE)\n    serve(session, args.ipc_dir)\n\n\nif __name__ == "__main__":\n    main()\n'
EMBEDDED_SOURCE_SHA256['scripts/colab_inference_worker.py'] = '182ab9296b8f6a03415752f7f6a77c1463f822719c5471b221ae5fc86be95086'


In [ ]:
# @title 编号与置信度展示函数（运行一次）
"""Render numbered SAM3-I instances and a confidence panel with Pillow only.

The caller supplies masks in exactly the same order as predictions. Instance
identifiers are one-based and remain the same in the overlay and score panel.
"""
from __future__ import annotations

import colorsys
import math

import numpy as np
from PIL import Image, ImageDraw, ImageFont


def instance_color(instance_id: int) -> tuple[int, int, int]:
    """Return a stable, bright RGB color for a one-based instance identifier."""
    if not isinstance(instance_id, int) or instance_id < 1:
        raise ValueError("instance_id must be a positive integer")
    hue = ((instance_id - 1) * 0.618033988749895 + 0.02) % 1.0
    return tuple(round(channel * 255) for channel in colorsys.hsv_to_rgb(hue, 0.78, 0.98))


def _font(size: int, bold: bool = False):
    names = [
        "/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf" if bold
        else "/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf",
        "DejaVuSans-Bold.ttf" if bold else "DejaVuSans.ttf",
        "/usr/share/fonts/truetype/liberation2/LiberationSans-Bold.ttf" if bold
        else "/usr/share/fonts/truetype/liberation2/LiberationSans-Regular.ttf",
    ]
    for name in names:
        try:
            return ImageFont.truetype(name, size)
        except OSError:
            continue
    return ImageFont.load_default(size=size)


def _validate_predictions(predictions):
    identifiers = []
    for prediction in predictions:
        instance_id = prediction["instance_id"]
        instance_color(instance_id)
        identifiers.append(instance_id)
        score = float(prediction["score"])
        if not math.isfinite(score) or not 0 <= score <= 1:
            raise ValueError("Prediction scores must be finite and in [0, 1].")
        box = prediction["box_xyxy"]
        if len(box) != 4 or not all(math.isfinite(float(value)) for value in box):
            raise ValueError("Prediction box_xyxy must contain four finite coordinates.")
    if len(set(identifiers)) != len(identifiers):
        raise ValueError("Instance identifiers must be unique.")


def _label_origin(box, badge_size, occupied, image_size):
    x0, y0, x1, y1 = box
    badge_width, badge_height = badge_size
    width, height = image_size
    candidates = [(x0, y0), (x1 - badge_width, y0),
                  (x0, y1 - badge_height), (x1 - badge_width, y1 - badge_height)]
    def clipped(candidate):
        return (max(0, min(width - badge_width, round(candidate[0]))),
                max(0, min(height - badge_height, round(candidate[1]))))
    def overlap(origin):
        x, y = origin
        return sum(max(0, min(x + badge_width, right) - max(x, left))
                   * max(0, min(y + badge_height, bottom) - max(y, top))
                   for left, top, right, bottom in occupied)
    origin = min((clipped(candidate) for candidate in candidates), key=overlap)
    occupied.append((origin[0], origin[1], origin[0] + badge_width, origin[1] + badge_height))
    return origin


def render_overlay(image: Image.Image, masks: np.ndarray, predictions: list[dict]) -> Image.Image:
    """Return original-size RGB image with translucent masks, boxes and IDs.

    No confidence values are drawn over the photograph; they belong in the panel
    returned by render_result_figure.
    """
    _validate_predictions(predictions)
    masks = np.asarray(masks)
    if masks.ndim == 4 and masks.shape[1] == 1:
        masks = masks[:, 0]
    width, height = image.size
    if masks.shape != (len(predictions), height, width):
        raise ValueError("Masks must have shape [K,H,W] or [K,1,H,W] matching the image.")
    overlay = image.convert("RGB")
    for mask, prediction in zip(masks, predictions):
        alpha = Image.fromarray(np.asarray(mask, dtype=np.uint8) * 88)
        overlay.paste(instance_color(prediction["instance_id"]), (0, 0, width, height), alpha)

    draw = ImageDraw.Draw(overlay)
    scale = width / 1200
    line_width = max(2, round(3 * scale))
    font_size = max(13, round(24 * scale))
    font = _font(font_size, bold=True)
    pad = max(3, round(5 * scale))
    occupied = []
    for prediction in predictions:
        color = instance_color(prediction["instance_id"])
        raw_box = [float(value) for value in prediction["box_xyxy"]]
        x0, x1 = sorted((max(0, min(width - 1, raw_box[0])),
                         max(0, min(width - 1, raw_box[2]))))
        y0, y1 = sorted((max(0, min(height - 1, raw_box[1])),
                         max(0, min(height - 1, raw_box[3]))))
        box = (x0, y0, x1, y1)
        draw.rectangle(box, outline=color, width=line_width)
        label = str(prediction["instance_id"])
        text_box = draw.textbbox((0, 0), label, font=font)
        badge_width = min(width, text_box[2] - text_box[0] + 2 * pad)
        badge_height = min(height, text_box[3] - text_box[1] + 2 * pad)
        x, y = _label_origin(box, (badge_width, badge_height), occupied, image.size)
        draw.rounded_rectangle((x, y, x + badge_width - 1, y + badge_height - 1),
                               radius=max(2, round(3 * scale)), fill=color,
                               outline=(255, 255, 255), width=max(1, round(scale)))
        # Dark text stays readable on the bright instance-color badge.
        draw.text((x + pad - text_box[0], y + pad - text_box[1]),
                  label, font=font, fill=(15, 23, 42))
    return overlay


def render_result_figure(
    overlay: Image.Image,
    predictions: list[dict],
    *,
    max_width: int = 1200,
) -> Image.Image:
    """Return a display-size RGB figure with the confidence panel below the image.

    The original-size overlay can be saved separately. This combined artifact
    uses at most max_width pixels across so its labels remain readable when
    displayed in Colab, including a score table with many instances.
    """
    _validate_predictions(predictions)
    if max_width < 160:
        raise ValueError("max_width must be at least 160 pixels.")
    width = min(max_width, max(320, overlay.width))
    height = max(1, round(overlay.height * width / overlay.width))
    photo = overlay.convert("RGB").resize((width, height), Image.Resampling.LANCZOS)
    font_size = max(12, min(18, round(width / 60)))
    body_font = _font(font_size)
    title_font = _font(font_size + 2, bold=True)
    pad = max(10, round(width / 60))
    title_height = font_size + 2 * pad
    columns = max(1, min(4, width // 270))
    row_height = font_size + 18
    rows = max(1, math.ceil(len(predictions) / columns))
    panel_height = title_height + rows * row_height + pad
    figure = Image.new("RGB", (width, height + panel_height), (248, 250, 252))
    figure.paste(photo, (0, 0))
    draw = ImageDraw.Draw(figure)
    draw.line((0, height, width, height), fill=(203, 213, 225), width=1)
    draw.text((pad, height + pad), "Instance confidence", fill=(15, 23, 42), font=title_font)
    if not predictions:
        draw.text((pad, height + title_height + 4), "No instances above the threshold.",
                  fill=(71, 85, 105), font=body_font)
        return figure
    column_width = (width - 2 * pad) / columns
    swatch_size = max(10, font_size - 2)
    for index, prediction in enumerate(predictions):
        row, column = divmod(index, columns)
        x = round(pad + column * column_width)
        y = height + title_height + row * row_height + 4
        color = instance_color(prediction["instance_id"])
        draw.rounded_rectangle((x, y + 3, x + swatch_size, y + 3 + swatch_size),
                               radius=2, fill=color)
        text = f"#{prediction['instance_id']}  {float(prediction['score']):.4f}"
        draw.text((x + swatch_size + 8, y), text, fill=(30, 41, 59), font=body_font)
    return figure

EMBEDDED_SOURCE_SHA256 = dict(globals().get("EMBEDDED_SOURCE_SHA256", {}))
EMBEDDED_SOURCE_SHA256['scripts/single_image_visualization.py'] = 'aeb91f0a3ae756acb552d61cdffb20c15ace08a4a46d03a43dee6ddb5189acd4'


In [ ]:
# @title 下载 GitHub 代码
import sys
import uuid

GITHUB_URL = "https://github.com/xheihex/SAM3-I.git" # @param {type:"string"}
GIT_REF = "main" # @param {type:"string"}
if globals().get("MODEL_WORKER") is not None:
    MODEL_WORKER.stop()
    MODEL_WORKER = None
DEPLOYMENT_ID = uuid.uuid4().hex[:12]
DEPLOYMENT_DIR = Path("/content/sam3i-deployments") / DEPLOYMENT_ID
LOGS = DEPLOYMENT_DIR / "logs"
REPO, SOURCE_INFO = checkout_github(GITHUB_URL, GIT_REF, DEPLOYMENT_DIR / "SAM3-I", LOGS)
SOURCE_INFO["notebook_version"] = "2026-11-03-resident-model-v2"
SOURCE_INFO["notebook_embedded_source_sha256"] = dict(EMBEDDED_SOURCE_SHA256)


## 2. 按 uv 锁文件安装，并记录实际 GPU 环境

使用独立 Python 3.12 环境；Colab 原生内核负责上传、Drive 和展示。子进程日志会同步打印并写入文件。

In [ ]:
# @title 创建环境并检查 GPU
run_logged([sys.executable, "-m", "pip", "install", "-q", "uv"], LOGS / "uv-install.log")
UV = [sys.executable, "-m", "uv"]
RUN_ENV = dict(os.environ, UV_CACHE_DIR="/content/.cache/sam3i-uv",
               HF_HOME="/content/.cache/sam3i-hf", MPLCONFIGDIR="/content/.cache/sam3i-matplotlib",
               PYTHONUNBUFFERED="1", CUDA_VISIBLE_DEVICES="0")
run_logged(UV + ["sync", "--locked", "--no-dev", "--project", str(REPO)],
           LOGS / "environment-sync.log", env=RUN_ENV)
RUN = UV + ["run", "--locked", "--no-dev", "--project", str(REPO), "python"]
ENVIRONMENT_PATH = DEPLOYMENT_DIR / "environment.json"
ENV_PROBE = """import json, platform, sys
from pathlib import Path
import torch
assert torch.cuda.is_available(), "Please enable a CUDA GPU runtime"
free, total = torch.cuda.mem_get_info()
info = {
    "python": platform.python_version(), "torch": torch.__version__, "cuda": torch.version.cuda,
    "gpu": torch.cuda.get_device_name(0), "capability": list(torch.cuda.get_device_capability(0)),
    "free_vram_gib": free / 2**30, "total_vram_gib": total / 2**30,
    "native_bf16": torch.cuda.is_bf16_supported(including_emulation=False),
}
Path(sys.argv[1]).write_text(json.dumps(info, indent=2), encoding="utf-8")
print(json.dumps(info, indent=2))
"""
run_logged(RUN + ["-c", ENV_PROBE, str(ENVIRONMENT_PATH)], LOGS / "gpu-environment.log", env=RUN_ENV)


## 3. 从 Google Drive 加载权重

保留你已运行成功的默认路径，可按实际位置修改。首次复制到 Colab 本地磁盘，再以内存映射读取；重新运行时，只有云盘来源路径、大小和修改时间都一致才复用本地副本。此检查用于避免重复复制，不等于权重内容校验。

In [ ]:
# @title 挂载 Drive 与准备权重
from google.colab import drive
drive.mount("/content/drive")

DRIVE_CHECKPOINT = "/content/drive/MyDrive/models/sam3-i/checkpoint.pt" # @param {type:"string"}
DRIVE_RESULTS = "/content/drive/MyDrive/results/SAM3-I" # @param {type:"string"}
CHECKPOINT = Path("/content/sam3i-checkpoints/checkpoint.pt")
if globals().get("MODEL_WORKER") is not None:
    MODEL_WORKER.stop()
    MODEL_WORKER = None
CHECKPOINT_INFO = ensure_local_checkpoint(DRIVE_CHECKPOINT, CHECKPOINT)
SOURCE_INFO["checkpoint_source"] = CHECKPOINT_INFO
print("Local checkpoint GiB:", round(CHECKPOINT.stat().st_size / 2**30, 3))


## 4. 检查 checkpoint 结构

仅本检查子进程隐藏 GPU，使用 meta 张量核对名称和形状；真实推理仍使用 GPU。

In [ ]:
# @title 审查权重
AUDIT_PATH = DEPLOYMENT_DIR / "checkpoint-audit.json"
run_logged(RUN + [
    str(REPO / "scripts/audit_single_image_checkpoint.py"),
    "--checkpoint", str(CHECKPOINT), "--output", str(AUDIT_PATH),
], LOGS / "checkpoint-audit.log", env=dict(RUN_ENV, CUDA_VISIBLE_DEVICES=""), echo=False)
audit = json.loads(AUDIT_PATH.read_text())
print("Audit OK:", audit["ok"])
print("Forward keys:", audit["matched_state_keys"], "/", audit["expected_state_keys"])
print("Loading keys:", audit["real_checkpoint_loading_topology"]["matched_state_keys"],
      "/", audit["real_checkpoint_loading_topology"]["state_keys"])
print("Audit CUDA initialized:", audit["cuda_initialized_after"])


## 5. 加载模型一次，后续复用显存

本单元格启动独立环境中的常驻 Python 进程，加载权重后保持模型在 GPU。相同配置下重复运行本单元格会复用已有进程。

后续只需反复运行推理单元格；更换图片时重新运行上传和推理即可。提示词、类别、阈值和计算精度可在每次推理时修改。每次查询重新计算图像特征，并创建独立提示状态；仅复用模型，不缓存图片特征。

重新下载代码、准备权重或停止进程后，需再运行本单元格。推理超时或中断会停止进程；验证错误可修正后重试，CUDA/显存错误后需重新加载。


In [ ]:
# @title 加载模型到显存（运行一次）
MODEL_WORKER = start_model_worker(
    REPO, CHECKPOINT, DEPLOYMENT_DIR / "runtime", LOGS,
    env=RUN_ENV, worker_source=COLAB_WORKER_SOURCE,
    existing=globals().get("MODEL_WORKER"),
)
SOURCE_INFO["resident_worker"] = dict(MODEL_WORKER.info)
print(json.dumps(MODEL_WORKER.info, ensure_ascii=False, indent=2))


## 6. 上传一张图片

换图时重新运行本单元格，再运行推理；无需重新加载模型。

In [ ]:
# @title 上传图片
from google.colab import files
from IPython.display import display
from PIL import Image as PILImage

uploaded_images = files.upload()
if len(uploaded_images) != 1:
    raise ValueError("请选择且只选择一张图片")
image_name, image_bytes = next(iter(uploaded_images.items()))
IMAGE_PATH = DEPLOYMENT_DIR / "input" / Path(image_name).name
IMAGE_PATH.parent.mkdir(exist_ok=True)
IMAGE_PATH.write_bytes(image_bytes)
del image_bytes, uploaded_images
with PILImage.open(IMAGE_PATH) as image:
    print("Stored image size (W,H):", image.size)
    preview = image.copy()
    preview.thumbnail((800, 800))
    display(preview)


## 7. 推理、编号展示，并自动保存到 Drive

| category | 选路 | 示例 |
|---|---:|---|
| concept | text_type=1 | shoe |
| simple | text_type=2 | the red shoe in the top row |
| complex | text_type=4 | the red item in the top row worn on feet while running |

`auto` 在原生 BF16 可用时使用 BF16，否则 FP16。此次运行记录只验证了 A100/BF16 的简单指令；其余路径的效果仍需测试。

编号按保存的实例顺序从 1 开始，框、掩码颜色、图下方分数与 JSON 中的 `instance_id` 一一对应。置信度是模型匹配分数，不代表已经校准的正确率。空预测会保留原图并显示空结果提示。

**本单元格结束时自动保存**结果 ZIP 至 Drive，包括编号总览图、原尺寸叠加图、各实例掩码、JSON、GPU 环境、实际 Git 提交、权重审查和日志。

重复运行下面单元格会使用同一模型进程；worker_id / worker_pid 保持一致，query_index 递增。加载耗时单独报告，每次推理耗时包含 CPU 掩码恢复，不包含模型加载、导出、展示和云盘保存。显存峰值每次重置统计，但包含常驻模型。

In [ ]:
# @title 单图推理、展示与自动保存（可反复运行）
PROMPT = "the red shoe in the top row" # @param {type:"string"}
CATEGORY = "simple" # @param ["concept", "simple", "complex"]
THRESHOLD = 0.4 # @param {type:"number"}
PRECISION = "auto" # @param ["auto", "bf16", "fp16", "fp32"]

if globals().get("MODEL_WORKER") is None or not MODEL_WORKER.is_alive():
    raise RuntimeError("请先运行「加载模型到显存」单元格")
OUTPUT = Path("/content/sam3i-results") / uuid.uuid4().hex[:12]
MODEL_WORKER.predict({
    "image": str(IMAGE_PATH), "prompt": PROMPT, "category": CATEGORY,
    "threshold": THRESHOLD, "precision": PRECISION, "output_dir": str(OUTPUT),
}, log_path=LOGS / f"inference-{OUTPUT.name}.log")
result, result_figure = render_saved_results(
    OUTPUT, IMAGE_PATH, SOURCE_INFO, render_overlay, render_result_figure,
)
summary = {
    "git_commit": SOURCE_INFO["git_commit"],
    "gpu": MODEL_WORKER.info["gpu"],
    "worker_id": result["worker_id"], "worker_pid": result["worker_pid"],
    "query_index": result["query_index"],
    "precision": result["precision"], "category": result["category"],
    "threshold": result["threshold"], "instances": result["num_instances"],
    "model_load_seconds_once": result["model_load_seconds"],
    "inference_seconds_excluding_model_load": result["inference_seconds"],
    "peak_pytorch_allocated_gib": result["peak_cuda_allocated_gib"],
    "confidence_by_instance": [
        {"instance_id": prediction["instance_id"], "score": prediction["score"]}
        for prediction in result["predictions"]
    ],
}
print(json.dumps(summary, ensure_ascii=False, indent=2))
display(result_figure)
RESULT_ZIP, SAVED_RESULT = archive_run(
    OUTPUT, SOURCE_INFO, AUDIT_PATH, LOGS, ENVIRONMENT_PATH, DRIVE_RESULTS,
)


## 8. 下载结果（可选）

结果已在上一步自动保存到 Drive；本步骤只下载到电脑。

In [ ]:
# @title 下载结果 ZIP
print("Drive:", SAVED_RESULT)
files.download(str(RESULT_ZIP))


展示图 `result_figure.png` 含下方置信度面板；`overlay.png` 保持原图尺寸；`result.json` 记录每个编号的 `score`、`box_xyxy`、掩码文件及 RLE。GitHub 的实际提交号与 notebook 展示代码哈希一起保存，便于复核。

资源说明见 [Colab FAQ](https://research.google.com/colaboratory/faq.html)、[Colab I/O](https://colab.research.google.com/notebooks/io.ipynb)、[uv 锁文件](https://docs.astral.sh/uv/concepts/projects/sync/)、[PyTorch AMP](https://docs.pytorch.org/docs/2.7/amp.html)。

## 9. 释放显存（可选）

模型会一直驻留显存直到停止进程或运行时结束。暂时不用时运行下面单元格；已保存的结果不受影响。

In [ ]:
# @title 释放模型显存（暂时不用时运行）
if globals().get("MODEL_WORKER") is not None:
    MODEL_WORKER.stop()
    MODEL_WORKER = None
print("模型进程已停止；再次推理前请重新运行加载模型单元格。")
